In [1]:
##  LightGBM + LSTM Blended Pipeline
##  12-Month Forward Return | Walk-Forward | Cross-sectional Z-Score
 
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from typing import Tuple, Dict
import warnings
warnings.filterwarnings('ignore')
 
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [2]:
import pandas as pd
import numpy as np
from typing import Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('fundamentals_with_prices.csv')


In [3]:

tickers = df['ticker'].unique().tolist()
print( len(tickers))

df.isna().sum()[df.isna().sum() > 0]
df = df.dropna(subset=["close_price"])

df.isna().sum()[df.isna().sum() > 0]
df = df.sort_values(["ticker", "fiscalDateEnding"])

df["future_return_1y"] = ( df.groupby("ticker")["close_price"].shift(-4) / df["close_price"] - 1)



99


In [4]:

# Must be .copy() — otherwise predict_df is a view and behaves unexpectedly
predict_df = df[df["future_return_1y"].isna()].copy()
df         = df[df["future_return_1y"].notna()].copy()



In [5]:
print(type(df))          # <class 'pandas.core.frame.DataFrame'>
print(type(predict_df))  # <class 'pandas.core.frame.DataFrame'>
print(f"df: {df.shape} | predict_df: {predict_df.shape}") # show first 3 elements to see what's inside

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
df: (1901, 87) | predict_df: (384, 87)


In [6]:
# ─────────────────────────────────────────────
# 3. QUARTER COLUMN
# ─────────────────────────────────────────────
df["fiscalDateEnding"]         = pd.to_datetime(df["fiscalDateEnding"])
df["quarter"]                  = df["fiscalDateEnding"].dt.to_period("Q")

predict_df["fiscalDateEnding"] = pd.to_datetime(predict_df["fiscalDateEnding"])
predict_df["quarter"]          = predict_df["fiscalDateEnding"].dt.to_period("Q")

# Verify
print(type(df), df.shape)
print(type(predict_df), predict_df.shape)
 

<class 'pandas.core.frame.DataFrame'> (1901, 88)
<class 'pandas.core.frame.DataFrame'> (384, 88)


In [7]:
tickers = df['ticker'].unique().tolist()
print( len(tickers))

96


In [8]:
# Add rank target (cross-sectional percentile per quarter)
df["return_rank"] = (df.groupby("quarter")["future_return_1y"]
                     .transform(lambda x: x.rank(pct=True)))

   # change target from "future_return_1y" to this


In [9]:
# ─────────────────────────────────────────────
# 4. FEATURE COLUMNS
# ─────────────────────────────────────────────
#target          = "future_return_1y"
target = "return_rank"
non_feature_cols = ["ticker", "fiscalDateEnding", "close_price",
                    "future_return_1y", "price_date", "quarter", "return_rank"]
 
feature_cols = df.columns.difference(non_feature_cols)
 
exclude_cols = ["ticker", "fiscalDateEnding", "close_price", "price_date", "quarter", "future_return_1y"]
missing_cols = [c for c in df.columns if "_missing" in c]
zscore_cols  = df.columns.difference(exclude_cols + missing_cols + [target])
 

In [10]:
# ─────────────────────────────────────────────
# 6. WALK-FORWARD SPLITS
# ─────────────────────────────────────────────
quarters = sorted(df["quarter"].unique())
test_q   = quarters[-1]

In [11]:

#train_stats = (df[df["quarter"] < test_q].groupby("quarter")[zscore_cols].agg(["mean", "std"]))

# Use expanding mean of train stats to normalize each quarter
#global_mean = train_stats['mean'].mean()
#global_std  = train_stats['std'].mean().replace(0, 1)

def normalize_quarter(grp):
    q = grp["quarter"].iloc[0]
    # Use only rows from quarters strictly before this one
    past = df[df["quarter"] < q][zscore_cols]
    if len(past) < 5:
        # Not enough history — fall back to within-quarter z-score
        mu  = grp[zscore_cols].mean()
        std = grp[zscore_cols].std().replace(0, 1)
    else:
        mu  = past.mean()
        std = past.std().replace(0, 1)
    grp = grp.copy()
    grp[zscore_cols] = (grp[zscore_cols] - mu) / std
    return grp

df = df.groupby("quarter", group_keys=False).apply(normalize_quarter)


In [ ]:
'''
# ─────────────────────────────────────────────
# 5. CROSS-SECTIONAL Z-SCORE (per quarter)
# ─────────────────────────────────────────────
def zscore(x):
    std = x.std()
    if std == 0 or np.isnan(std):
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std
 
df[zscore_cols] = df.groupby("quarter")[zscore_cols].transform(zscore)
 
# Apply same z-score to predict_df using training stats to avoid leakage
train_stats = df.groupby("quarter")[zscore_cols].agg(["mean", "std"])
'''

In [12]:

 
splits = []
for i in range(8, len(quarters) - 1):
    train_qs = quarters[:i]
    val_q    = quarters[i]
    train_df = df[df["quarter"].isin(train_qs)]
    val_df   = df[df["quarter"] == val_q]
    splits.append((train_df, val_df))
 
test_df = df[df["quarter"] == test_q].copy()
 
print(f"Quarters: {len(quarters)} | Walk-forward folds: {len(splits)} | Test quarter: {test_q}")
 

Quarters: 21 | Walk-forward folds: 12 | Test quarter: 2025Q1


In [13]:
print("\n── Training LightGBM ──")
lgbm_models = []
lgbm_val_ics  = []
lstm_val_ics  = []

for train_df, val_df in splits:
    
    train_df = train_df.copy()
    val_df   = val_df.copy()

    # normalize using only training data
    mu  = train_df[zscore_cols].mean()
    std = train_df[zscore_cols].std().replace(0,1)

    train_df[zscore_cols] = (train_df[zscore_cols] - mu) / std
    val_df[zscore_cols]   = (val_df[zscore_cols] - mu) / std

    X_train = train_df[feature_cols]
    y_train = train_df[target]

    X_val = val_df[feature_cols]
    y_val = val_df[target]

    model = lgb.LGBMRegressor( n_estimators=300,
                        learning_rate=0.03,
                        max_depth=4,
                        num_leaves=20,
                        min_child_samples=10,
                        subsample=0.7,
                        colsample_bytree=0.7,
                        reg_alpha=1.0,
                        reg_lambda=2.0,
                        random_state=42,
                        verbosity=-1
                    )

    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)])

    lgbm_models.append(model)

    # Track LightGBM val IC
    lgbm_val_pred    = model.predict(X_val)
    lgbm_ic, _       = spearmanr(y_val, lgbm_val_pred)
    lgbm_val_ics.append(lgbm_ic)


── Training LightGBM ──


In [15]:
# LightGBM predictions on test
test_df = test_df.copy()
test_df[zscore_cols] = (test_df[zscore_cols] - mu) / std
lgbm_preds_test = np.mean([m.predict(test_df[feature_cols]) for m in lgbm_models], axis=0)
 


In [16]:

# LightGBM predictions on real predict_df
predict_df[zscore_cols] = (predict_df[zscore_cols] - mu) / std

# Predict
lgbm_preds_real = np.mean(
    [m.predict(predict_df[feature_cols]) for m in lgbm_models],
    axis=0
)


In [18]:
# ─────────────────────────────────────────────
# 8. LSTM — SEQUENCE BUILDER
# ─────────────────────────────────────────────
SEQ_LEN = 4  # 4 quarters = 1 year of history per sample
 
def build_sequences(data: pd.DataFrame, feat_cols, seq_len: int = 4):
    """Build (samples, timesteps, features) for LSTM, one sequence per ticker-quarter."""
    X_list, y_list, meta_list = [], [], []
    for ticker, grp in data.groupby("ticker"):
        grp  = grp.sort_values("fiscalDateEnding").reset_index(drop=True)
        vals = grp[feat_cols].values.astype(np.float32)
        lbls = grp[target].values
        qdts = grp["quarter"].values
        for i in range(seq_len - 1, len(grp)):
            seq = vals[i - seq_len + 1 : i + 1]
            if not np.isnan(seq).any():
                X_list.append(seq)
                y_list.append(lbls[i])
                meta_list.append({"ticker": ticker, "quarter": qdts[i]})
    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.float32),
            pd.DataFrame(meta_list))
 
def build_sequences_no_label(data: pd.DataFrame, feat_cols, seq_len: int = 4):
    """Same as above but for predict_df (no label column)."""
    X_list, meta_list = [], []
    for ticker, grp in data.groupby("ticker"):
        grp  = grp.sort_values("fiscalDateEnding").reset_index(drop=True)
        vals = grp[feat_cols].values.astype(np.float32)
        qdts = grp["quarter"].values
        for i in range(seq_len - 1, len(grp)):
            seq = vals[i - seq_len + 1 : i + 1]
            if not np.isnan(seq).any():
                X_list.append(seq)
                meta_list.append({"ticker": ticker, "quarter": qdts[i]})
    return np.array(X_list, dtype=np.float32), pd.DataFrame(meta_list)
 

In [19]:

# Build sequences from labeled data
X_all, y_all, meta_all = build_sequences(df, feature_cols, SEQ_LEN)
print(f"\nLSTM sequences built: {X_all.shape}  (samples × timesteps × features)")



LSTM sequences built: (1251, 4, 82)  (samples × timesteps × features)


In [20]:
# ─────────────────────────────────────────────
# 9. LSTM — MODEL DEFINITION
# ─────────────────────────────────────────────
def build_lstm(seq_len: int, n_features: int) -> tf.keras.Model:
    """
    Small LSTM suited for ~1,800 samples.
    Key: fewer units, aggressive dropout, no deep stacking.
    """
    inp = Input(shape=(seq_len, n_features))
    x   = LSTM(32, return_sequences=True)(inp)   # small hidden size
    x   = Dropout(0.4)(x)
    x   = LSTM(16)(x)
    x   = BatchNormalization()(x)
    x   = Dropout(0.4)(x)
    x   = Dense(16, activation="relu")(x)
    out = Dense(1, activation="linear")(x)       # regression output
    model = Model(inp, out)
    model.compile(optimizer=Adam(1e-3), loss="mse")
    return model
    

In [33]:
lstm_models = []
SEQ_LEN = 4

for i in range(8, len(quarters) - 1):

    train_qs = set(quarters[:i])
    val_q_   = quarters[i]

    train_df_fold = df[df["quarter"].isin(train_qs)].copy()

    val_q_index = quarters.index(val_q_)
    val_hist_qs = quarters[max(0, val_q_index - SEQ_LEN + 1): val_q_index + 1]

    val_df_fold = df[df["quarter"].isin(val_hist_qs)].copy()

    # Normalize using train stats
    mu  = train_df_fold[zscore_cols].mean()
    std = train_df_fold[zscore_cols].std().replace(0,1)

    train_df_fold[zscore_cols] = (train_df_fold[zscore_cols] - mu) / std
    val_df_fold[zscore_cols]   = (val_df_fold[zscore_cols] - mu) / std

    X_tr, y_tr, _   = build_sequences(train_df_fold, feature_cols, SEQ_LEN)
    X_val_, y_val_, _ = build_sequences(val_df_fold, feature_cols, SEQ_LEN)

    if len(X_tr) == 0 or len(X_val_) == 0:
        print("Skipping fold due to no sequences")
        continue

    model = build_lstm(SEQ_LEN, X_tr.shape[2])

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val_, y_val_),
        epochs=100,
        batch_size=32,
        callbacks=[EarlyStopping(monitor="val_loss", patience=10,
                                 restore_best_weights=True)],
        verbose=0
    )

    lstm_models.append(model)

    print(f"Fold {i-7}/{len(quarters)-9}: train={len(X_tr)}, val={len(X_val_)}")

    lstm_val_pred = model.predict(X_val_, verbose=0).flatten()
    lstm_ic, _ = spearmanr(y_val_, lstm_val_pred)

    lstm_val_ics.append(lstm_ic)

Fold 1/12: train=346, val=74
Fold 2/12: train=420, val=71
Fold 3/12: train=491, val=68
Fold 4/12: train=559, val=71
Fold 5/12: train=630, val=71
Fold 6/12: train=701, val=75
Fold 7/12: train=776, val=79
Fold 8/12: train=855, val=79
Fold 9/12: train=934, val=81
Fold 10/12: train=1015, val=80
Fold 11/12: train=1095, val=76
Fold 12/12: train=1171, val=72


In [34]:
# ── Summary ───────────────────────────────────────────────────────────
print(f"\n  LightGBM IC per fold: {[round(x,4) for x in lgbm_val_ics]}")
print(f"  LSTM IC per fold    : {[round(x,4) for x in lstm_val_ics]}")
print(f"\n  LightGBM avg IC (last 4): {np.mean(lgbm_val_ics[-4:]):+.4f}")
print(f"  LSTM     avg IC (last 4): {np.mean(lstm_val_ics[-4:]):+.4f}")


  LightGBM IC per fold: [0.5329, 0.3085, 0.4059, 0.4314, 0.53, 0.6328, 0.47, 0.3712, 0.3128, 0.464, 0.4003, 0.3397]
  LSTM IC per fold    : [0.5122, 0.0434, -0.1657, 0.2515, 0.3858, 0.2824, 0.3154, 0.1787, 0.2153, 0.3305, 0.2954, 0.2471]

  LightGBM avg IC (last 4): +0.3792
  LSTM     avg IC (last 4): +0.2721


In [35]:
# ─────────────────────────────────────────────
# 11. LSTM PREDICTIONS — TEST SET
# ─────────────────────────────────────────────
test_mask_lstm = meta_all["quarter"] == test_q
X_test_lstm    = X_all[test_mask_lstm]
meta_test_lstm = meta_all[test_mask_lstm].reset_index(drop=True)
 
lstm_preds_test = np.mean([m.predict(X_test_lstm, verbose=0).flatten() for m in lstm_models], axis=0)


In [36]:
# ── Safe weight definition ─────────────────────────────────────────────

def safe_ic_weight(ic_list_name, fallback_ic):
    """Get blend weight safely — works even if variable was never defined."""
    ic_list = globals().get(ic_list_name, [])
    if len(ic_list) == 0:
        print(f"  {ic_list_name} not found — using fallback IC: {fallback_ic:.4f}")
        return fallback_ic
    recent_ic = np.mean([max(ic, 0) for ic in ic_list[-4:]])
    print(f"  {ic_list_name}: last 4 fold ICs = {[round(x,4) for x in ic_list[-4:]]}")
    print(f"  {ic_list_name}: avg (floored at 0) = {recent_ic:.4f}")
    return recent_ic

recent_lgbm_ic = safe_ic_weight("lgbm_val_ics", fallback_ic=0.0)
recent_lstm_ic = safe_ic_weight("lstm_val_ics",  fallback_ic=0.3333)

total  = recent_lgbm_ic + recent_lstm_ic + 1e-9
w_lgbm = recent_lgbm_ic / total
w_lstm  = recent_lstm_ic / total

print(f"\n  Final blend weights:")
print(f"  LightGBM : {w_lgbm:.1%}")
print(f"  LSTM     : {w_lstm:.1%}")

  lgbm_val_ics: last 4 fold ICs = [0.3128, 0.464, 0.4003, 0.3397]
  lgbm_val_ics: avg (floored at 0) = 0.3792
  lstm_val_ics: last 4 fold ICs = [0.2153, 0.3305, 0.2954, 0.2471]
  lstm_val_ics: avg (floored at 0) = 0.2721

  Final blend weights:
  LightGBM : 58.2%
  LSTM     : 41.8%


In [37]:
# ─────────────────────────────────────────────
# 12. BLEND — TEST SET
# ─────────────────────────────────────────────
# LightGBM operates on rows, LSTM on sequences → align by ticker
lgbm_test_df = test_df[["ticker", "quarter", target]].copy()
lgbm_test_df["lgbm_pred"] = lgbm_preds_test
 
lstm_test_df = meta_test_lstm.copy()
lstm_test_df["lstm_pred"] = lstm_preds_test
 
blended_test = lgbm_test_df.merge(lstm_test_df, on=["ticker", "quarter"], how="inner")
blended_test["blended_pred"] = 0.5 * blended_test["lgbm_pred"] + 0.5 * blended_test["lstm_pred"]
 

In [38]:
# ─────────────────────────────────────────────
# 13. EVALUATE — SPEARMAN IC
# ─────────────────────────────────────────────
def spearman_ic(actual, predicted, label=""):
    corr, _ = spearmanr(actual, predicted)
    print(f"  Spearman IC {label:15s}: {corr:.4f}")
    return corr
 
print(f"\n── Evaluation on test quarter: {test_q} ──")
spearman_ic(blended_test[target], blended_test["lgbm_pred"],  label="LightGBM")
spearman_ic(blended_test[target], blended_test["lstm_pred"],  label="LSTM")
spearman_ic(blended_test[target], blended_test["blended_pred"], label="Blended (50/50)")
 
# Top-20 from test (ranked by blended score)
top20_test = (blended_test
    .sort_values("blended_pred", ascending=False)
    .head(20)[["ticker", "quarter", "lgbm_pred", "lstm_pred", "blended_pred", target]])
 
print(f"\nTop 20 predicted performers — {test_q}")
print(top20_test.to_string(index=False))



── Evaluation on test quarter: 2025Q1 ──
  Spearman IC LightGBM       : 0.3333
  Spearman IC LSTM           : 0.4048
  Spearman IC Blended (50/50): 0.4524

Top 20 predicted performers — 2025Q1
ticker quarter  lgbm_pred  lstm_pred  blended_pred  return_rank
  PANW  2025Q1   0.468016   0.554865      0.511441     0.444444
  AMAT  2025Q1   0.503770   0.481226      0.492498     1.000000
   ADI  2025Q1   0.458876   0.433091      0.445983     0.888889
  CSCO  2025Q1   0.405532   0.451169      0.428351     0.777778
  WDAY  2025Q1   0.420055   0.416969      0.418512     0.111111
  CPRT  2025Q1   0.417883   0.414190      0.416036     0.222222
  SNPS  2025Q1   0.369376   0.436562      0.402969     0.333333
    ZM  2025Q1   0.335602   0.412548      0.374075     0.666667


In [39]:
# ── Real predictions (no label) ──
print("\n── Real predictions (no label) ──")

# Normalize predict_df using stats from ALL labeled data (df)
# since predict_df quarters come AFTER all training quarters
def normalize_predict(grp):
    # Use the entire labeled df as "past" — it all precedes predict_df
    past = df[zscore_cols]
    mu   = past.mean()
    std  = past.std().replace(0, 1)
    grp  = grp.copy()
    grp[zscore_cols] = (grp[zscore_cols] - mu) / std
    return grp

predict_df = predict_df.groupby("quarter", group_keys=False).apply(normalize_predict)

# LightGBM on predict_df
lgbm_preds_real_df = predict_df[["ticker", "quarter"]].copy()
lgbm_preds_real_df["lgbm_pred"] = np.mean(
    [m.predict(predict_df[feature_cols]) for m in lgbm_models], axis=0
)

# LSTM on predict_df
X_real, meta_real = build_sequences_no_label(predict_df, feature_cols, SEQ_LEN)

if len(X_real) > 0:
    lstm_preds_real = np.mean(
        [m.predict(X_real, verbose=0).flatten() for m in lstm_models], axis=0
    )
    meta_real["lstm_pred"] = lstm_preds_real
    blended_real = lgbm_preds_real_df.merge(meta_real, on=["ticker", "quarter"], how="inner")
    blended_real["blended_pred"] = w_lgbm * blended_real["lgbm_pred"] + w_lstm * blended_real["lstm_pred"]
else:
    print("  Not enough history for LSTM sequences — using LightGBM only")
    blended_real = lgbm_preds_real_df.copy()
    blended_real["lstm_pred"]    = np.nan
    blended_real["blended_pred"] = blended_real["lgbm_pred"]

# Top 20 per quarter
last_two_q = sorted(blended_real["quarter"].unique())[-2:]
top20_final = (
    blended_real[blended_real["quarter"].isin(last_two_q)]
    .sort_values(["quarter", "blended_pred"], ascending=[False, False])
    .groupby("quarter")
    .head(20)
)
print(f"\nTop 20 predicted performers — last 2 quarters")
print(top20_final[["ticker", "quarter", "lgbm_pred", "lstm_pred", "blended_pred"]].to_string(index=False))



── Real predictions (no label) ──

Top 20 predicted performers — last 2 quarters
ticker quarter  lgbm_pred  lstm_pred  blended_pred
  SNPS  2026Q1   0.620967   0.659989      0.637268
  CPRT  2026Q1   0.578329   0.662193      0.613362
  AMAT  2026Q1   0.574562   0.658990      0.609831
   ADI  2026Q1   0.574773   0.647455      0.605135
  CSCO  2026Q1   0.561417   0.658802      0.602098
  WDAY  2026Q1   0.550473   0.652508      0.593097
  PANW  2026Q1   0.496211   0.660145      0.564692
    ZM  2026Q1   0.504388   0.628401      0.556193
  MELI  2025Q4   0.663085   0.663949      0.663446
  TSLA  2025Q4   0.656979   0.658690      0.657694
  MDLZ  2025Q4   0.651852   0.657957      0.654402
   PEP  2025Q4   0.625731   0.654889      0.637912
  DDOG  2025Q4   0.628667   0.648286      0.636863
   HON  2025Q4   0.614325   0.660000      0.633405
  OKTA  2025Q4   0.626146   0.637589      0.630926
  ODFL  2025Q4   0.545658   0.745381      0.629090
  PCAR  2025Q4   0.602073   0.665897      0.628734


In [40]:
# ─────────────────────────────────────────────
# 15. RANK BASED ON BLENDED PREDICTION
# ─────────────────────────────────────────────
# Add this block right after Section 14 in your blended pipeline

# ── 15a. Cross-sectional rank within each quarter ─────────────────────
# pct=True gives percentile rank 0→1 (1 = best predicted performer)
# This is the standard quant approach — rank within universe per period

for frame, label in [(blended_test, "test"), (blended_real, "real")]:

    frame["rank"]       = frame.groupby("quarter")["blended_pred"].rank(ascending=True, pct=True)
    frame["rank_int"]   = frame.groupby("quarter")["blended_pred"].rank(ascending=False, method="min").astype(int)
    frame["decile"]     = pd.qcut(frame["rank"], q=10, labels=False) + 1   # 1=bottom, 10=top
    frame["top20pct"]   = (frame["rank"] >= 0.80).astype(int)              # 1 = top 20%


# ── 15b. Test set ranking table ───────────────────────────────────────
print(f"\n── Ranked predictions — test quarter: {test_q} ──")

rank_cols_test = ["rank_int", "ticker", "quarter",
                  "lgbm_pred", "lstm_pred", "blended_pred",
                  "rank", "decile", "top20pct", target]

ranked_test = (blended_test[rank_cols_test]
               .sort_values("rank_int")
               .reset_index(drop=True))

print(ranked_test.to_string(index=False))


# ── 15c. Top 20% picks from test (validated — we know the outcome) ────
top20pct_test = ranked_test[ranked_test["top20pct"] == 1].copy()

print(f"\n── Top 20% picks ({len(top20pct_test)} stocks) — {test_q} ──")
print(top20pct_test[["rank_int", "ticker", "blended_pred", "rank", target]].to_string(index=False))

# Hit rate: what fraction of our top 20% picks actually ended up in top 20% by real return
actual_top20pct = ranked_test[target].quantile(0.80)
top20pct_test["was_correct"] = (top20pct_test[target] >= actual_top20pct).astype(int)
hit_rate = top20pct_test["was_correct"].mean()
print(f"\n  Hit rate (predicted top 20% that were actually top 20%): {hit_rate:.1%}")


# ── 15d. Real prediction ranking (no label — future picks) ────────────
print(f"\n── Ranked real predictions — last 2 quarters ──")

rank_cols_real = ["rank_int", "ticker", "quarter",
                  "lgbm_pred", "lstm_pred", "blended_pred",
                  "rank", "decile", "top20pct"]

for q in sorted(blended_real["quarter"].unique())[-2:]:
    q_df = blended_real[blended_real["quarter"] == q].copy()

    # Re-rank within this quarter only
    q_df["rank"]     = q_df["blended_pred"].rank(ascending=True, pct=True)
    q_df["rank_int"] = q_df["blended_pred"].rank(ascending=False, method="min").astype(int)
    q_df["decile"]   = pd.qcut(q_df["rank"], q=10, labels=False) + 1
    q_df["top20pct"] = (q_df["rank"] >= 0.80).astype(int)

    print(f"\n  Quarter: {q}  |  Universe: {len(q_df)} stocks")
    print(f"  {'Rank':<6} {'Ticker':<12} {'Blended':>10} {'Percentile':>12} {'Decile':>8} {'Top20%':>8}")
    print(f"  {'─'*6} {'─'*12} {'─'*10} {'─'*12} {'─'*8} {'─'*8}")

    for _, row in q_df.sort_values("rank_int").iterrows():
        marker = " ◀ BUY" if row["top20pct"] == 1 else ""
        print(f"  {int(row['rank_int']):<6} {row['ticker']:<12} "
              f"{row['blended_pred']:>10.4f} "
              f"{row['rank']:>11.1%} "
              f"{int(row['decile']):>8}"
              f"{int(row['top20pct']):>8}"
              f"{marker}")


# ── 15e. Summary stats per decile (test set only — has real returns) ──
print(f"\n── Decile analysis — test quarter {test_q} ──")
print(f"  (Does higher predicted decile → higher actual return?)\n")

decile_summary = (ranked_test.groupby("decile")
    .agg(
        n_stocks       = (target, "count"),
        avg_pred       = ("blended_pred", "mean"),
        avg_actual_ret = (target, "mean"),
        hit_rate_top20 = ("top20pct", lambda x:
                          (ranked_test.loc[x.index, target] >=
                           ranked_test[target].quantile(0.8)).mean())
    )
    .reset_index()
)

print(f"  {'Decile':>7} {'Stocks':>7} {'Avg pred':>10} {'Avg actual return':>18} {'Hit rate':>10}")
print(f"  {'─'*7} {'─'*7} {'─'*10} {'─'*18} {'─'*10}")
for _, row in decile_summary.iterrows():
    bar = "█" * int(max(0, row["avg_actual_ret"]) * 100)
    print(f"  {int(row['decile']):>7} {int(row['n_stocks']):>7} "
          f"{row['avg_pred']:>10.4f} "
          f"{row['avg_actual_ret']:>17.1%}  "
          f"{row['hit_rate_top20']:>9.1%}  {bar}")

print(f"\n  Interpretation: decile 10 should have the highest avg actual return")
print(f"  if the model has genuine predictive power (positive IC).")


── Ranked predictions — test quarter: 2025Q1 ──
 rank_int ticker quarter  lgbm_pred  lstm_pred  blended_pred  rank  decile  top20pct  return_rank
        1   PANW  2025Q1   0.468016   0.554865      0.511441 1.000      10         1     0.444444
        2   AMAT  2025Q1   0.503770   0.481226      0.492498 0.875       9         1     1.000000
        3    ADI  2025Q1   0.458876   0.433091      0.445983 0.750       8         0     0.888889
        4   CSCO  2025Q1   0.405532   0.451169      0.428351 0.625       6         0     0.777778
        5   WDAY  2025Q1   0.420055   0.416969      0.418512 0.500       5         0     0.111111
        6   CPRT  2025Q1   0.417883   0.414190      0.416036 0.375       3         0     0.222222
        7   SNPS  2025Q1   0.369376   0.436562      0.402969 0.250       2         0     0.333333
        8     ZM  2025Q1   0.335602   0.412548      0.374075 0.125       1         0     0.666667

── Top 20% picks (2 stocks) — 2025Q1 ──
 rank_int ticker  blended_pr